# **Mount Drive**

# **Remove 5% & 95% Areas**

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_agg_irig.csv'
df_fail = pd.read_csv(ffail)
df_fail = df_fail.drop(columns=['Unnamed: 0'])
df_fail = df_fail.sort_values(by=['State', 'County'])
# cond = df_fail['year']>2011
# df_fail = df_fail[cond]
df_fail.iloc[0:5,:]

In [ ]:
cond1 = df_fail['Irrigation Practice'] == 'ALL'
cond2 = df_fail['Planted Acres'] > 0
percentiles = df_fail[cond1 & cond2].groupby('Crop').quantile([0.05, 0.95])['Planted Acres']
percentiles = percentiles.reset_index()
percentiles = percentiles.pivot(index='Crop', columns='level_1', values='Planted Acres')
percentiles = percentiles.rename(columns = {0.05:'P05',0.95:'P95'}).reset_index()
percentiles

In [ ]:
cond1 = df_fail['Irrigation Practice'] == 'ALL'
df_fail_agg = df_fail[cond1].groupby(['FIPS','Crop','year']).sum().reset_index()
df_fail_agg = pd.merge(df_fail_agg,percentiles,on=['Crop'])

cond1 = df_fail_agg['Planted Acres'] > df_fail_agg['P05']
cond2 = df_fail_agg['Planted Acres'] < df_fail_agg['P95']
df_fail_agg = df_fail_agg[cond1 & cond2]
df_fail_agg = df_fail_agg.drop(columns=['Planted Acres','Prevented Acres','Failed Acres','P05','P95'])
df_fail_agg
df_fail = pd.merge(df_fail,df_fail_agg,on=['FIPS','Crop','year'])


In [ ]:
ffail = '<DATA_ROOT>/crop_failure_data/crops_failure_0595.csv'
df_fail.to_csv(ffail)

In [ ]:
df_fail

# New Section

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import geopandas as gpd
from matplotlib import pyplot as plt
import seaborn as sns
from glob import glob

In [ ]:
import pandas as pd
fpath1 = '<DATA_ROOT>/WeatherIndex/Tercile_temperature.csv'
fpath2 = '<DATA_ROOT>/WeatherIndex/Tercile_drought_old.csv'
dft = pd.read_csv(fpath1)
dfd = pd.read_csv(fpath2)

In [ ]:
dfd = dfd.rename(columns={'Year':'year', 'STATE':'State', 'COUNTY':'County'})
fpath = '<DATA_ROOT>/WeatherIndex/svi2020fips.csv'
df_fips_obj = pd.read_csv(fpath)[['FIPS',	'OBJECTID']]
dfd = pd.merge(dfd,df_fips_obj,on=['OBJECTID'])
dfd.iloc[0:5,:]

In [ ]:
import pandas as pd
fsvi = '<DATA_ROOT>/SVI/SVI_timeseries.csv'
df_svi = pd.read_csv(fsvi)
df_svi = df_svi.sort_values(by=['State', 'County'])
df_svi.columns

In [ ]:
df_fail = df_fail_quant
df_fail.columns

In [ ]:
# temps_ths = [
#      'Thr26', 'Thr27', 'Thr28', 'Thr29', 'Thr30', 'Thr31', 'Thr32',
#      'Thr33', 'Thr34', 'Thr35', 'Thr36' ]

# drought_ths = ['spei14d', 'spei30d', 'spei90d', 'spi14d', 'spi30d', 'spi90d']
drought_ths = ['PDSI', 'SPEI3', 'SPEI6', 'SPEI9', 'SPEI12', 'SPI3','SPI6', 'SPI9', 'SPI12']

temps_cls = [-1,0,1]

join_cols = ['FIPS', 'year']
df_fail = pd.merge(df_fail,dfd.drop(columns=['State',	'County']),on=join_cols)
df_fail

In [ ]:
crops = ['CORN', 'COTTON', 'OATS', 'SORGHUM', 'SOYBEANS', 'WHEAT', 'BARLEY', 'HAY']

irig = ['I' , 'N' , 'ALL']
irig_conf = {
    irig[0]: ['Irrigation','#0000ff'],
    irig[1]: ['Non-Irrigation', '#ff9900'],
    irig[2]: ['All Irrigation Types','#00ff00']
}

#
# themes = ['RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4']
themes = ['RPL_THEME1','RPL_THEME2','RPL_THEME3','RPL_THEME4']

theme_colors = {
    themes[0]: '#2c7bb6',    themes[1]: '#abd9e9',
    themes[2]: '#fdae61',    themes[3]: '#d7191c'
}

fail_share_th = 0.2

df_fail['fail_share'] = df_fail['Failed Acres'] /  df_fail['Planted Acres']

for crop in crops:
  for theme in themes:
    bins = [0, 0.25, 0.5, 0.75, 1]
    labels = [1, 2, 3, 4]
    df_svi['cat_theme'] = pd.cut(df_svi[theme], bins=bins, labels=labels)

    for th in drought_ths:
      fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharey=True)
      i = 0

      for ig in irig:
        cond0 = (df_fail['fail_share'] > 0)
        cond1 = (df_fail['fail_share'] < fail_share_th)
        cond2 = (df_fail['Crop'] == crop)
        cond3 = (df_fail['Irrigation Practice'] == ig)
        df_fail_gt0 = df_fail[(cond0) & (cond1) & (cond2) & (cond3)]

        join_cols = ['State', 'County', 'year']
        df_merge = pd.merge(df_fail_gt0,df_svi,on=join_cols,how='inner')

        for tcls in temps_cls:
          cond_th = df_merge[th] == tcls
          df_merge_th = df_merge[cond_th]

          ax = axes[i//3,i%3]
          i = i + 1
          sns.boxplot(x='cat_theme', y='fail_share', data=df_merge_th, ax=ax, whis=[1,99])
          sns.swarmplot(x="cat_theme", y="fail_share", data=df_merge_th, size=3 ,color="0.15", ax=ax)

          ax.set_title(f'{crop} {theme} {irig_conf.get(ig)[0]} {th}:{tcls}',
                       fontdict={'fontsize':9})
          ax.set_xlabel('')
          ax.set_xticklabels([])
          ylim = (df_fail_gt0.fail_share.min(), df_fail_gt0.fail_share.max())
          ax.set_ylim(ylim)

        # out_path = '<DATA_ROOT>/ProcessedData/20231104-CropTemperatureThr-violinplot-fth01/'
        out_path = '<DATA_ROOT>/ProcessedData/20231106-CropDroughtold-boxplots_swarm/'
      plt.savefig(out_path + crop + '_' + theme + '_' + th + '.jpg' , dpi=400)

  #   plt.tight_layout()
  #   plt.show()
  #   break
  # break
  #




In [ ]:
df_merge

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create a sample DataFrame
df = df_fail

# Create a histogram of the 'Age' column
plt.hist(df['fail_share'], bins=10, color='blue', edgecolor='black')
plt.xlabel('fail_share')
plt.ylabel('Frequency')
plt.title('fail_share Distribution')
plt.show()
